In [13]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:90% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

# 1. tensorflow v2.xx에서 v1 사용하기

In [5]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior() # tensorflow v2 비활성화하고 v1만 활성화
import numpy as np
import pandas as pd

Instructions for updating:
non-resource variables are not supported in the long term


## Tensorflow
- 데이터 흐름 그래프(tensor 흐름을 나타내는 설계도)를 사용하는 수치 계산 라이브러리
- 그래프는 node(데이터, 연산)와 edge로 구성
- sess = tf.Session()을 이용하여 실행환경 
- sess.run()을 통해서 실행결과를 확인

In [6]:
# 1. tensor(상수 node, 변수 node, 연산 node) 정의
node1 = tf.constant('Hello, Tensorflow')
# 2. 세션 생성(연산을 실행하는 환경 생성)
sess = tf.Session()
# 3. 실행
print(sess.run(node1))
print(sess.run(node1).decode())

b'Hello, Tensorflow'
Hello, Tensorflow


In [8]:
# 간단한 연산 tensor 그래프
# 1. 그래프 정의
node1 = tf.constant(10, dtype=tf.float16)
node2 = tf.constant(20, dtype=tf.float16)
node3 = tf.add(node1, node2)
# 2. 세션 생성
sess = tf.Session()
# 3. 세션 실행 및 결과
n1, n2, n3 = sess.run([node1, node2, node3])
print(n1, n2, n3)

10.0 20.0 30.0


In [11]:
# 타입 변경
node1 = tf.constant(np.array([10,20,30]), dtype=tf.int16)
node2 = tf.cast(node1, dtype=tf.float32)
sess = tf.Session()
print(sess.run( [node1, node2] ))

[array([10, 20, 30], dtype=int16), array([10., 20., 30.], dtype=float32)]


In [17]:
# 평균값 계산 : tf.reduce_mean()
data = np.array([1., 2, 3, 4])
m = tf.reduce_mean(data)
sess = tf.Session()
sess.run(m)

2.5

In [19]:
# tf.random_normal([shape]) : 평균 0, 표준편차는 1인 shape 난수 배열. 기본적으로 float32
w = tf.random.normal([1,3])
sess = tf.Session()
sess.run(w)

array([-1.2183166], dtype=float32)

In [22]:
# 변수노드
w = tf.Variable( tf.random_normal([1]) )
sess = tf.Session()
sess.run(tf.global_variables_initializer()) # 난수가 발생될 변수 초기화
sess.run(w)

array([0.09721699], dtype=float32)

# 2.tensorflow v1을 이용한 회귀분석 구현
## 2.1 독립(입력)변수 x가 1개, 종속(타겟)변수 y가 1개

In [39]:
# tensor 그래프 정의
# 데이터 셋 확보
x = np.array([1,2,3])
y = np.array([2,3,4])
# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight')
b = tf.Variable( tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습 목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GrandientDesent)
'''
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = optimizer.minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 6001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b])
    if step%300==1:
        print(f'{step}번째 cost:{cost_val}, w:{w_val}, b:{b_val}')
print(f'{step}번째 cost:{cost_val}, w:{w_val}, b:{b_val}')

1번째 cost:7.43715238571167, w:[1.1196822], b:[-1.6940409]
301번째 cost:0.18014074862003326, w:[1.4917637], b:[-0.11789482]
601번째 cost:0.04250591993331909, w:[1.2388777], b:[0.45697516]
901번째 cost:0.0100296875461936, w:[1.1160364], b:[0.7362219]
1201번째 cost:0.0023665782064199448, w:[1.0563653], b:[0.8718687]
1501번째 cost:0.0005584166501648724, w:[1.0273798], b:[0.93775934]
1801번째 cost:0.00013176283391658217, w:[1.0132998], b:[0.96976626]
2101번째 cost:3.109109457000159e-05, w:[1.0064604], b:[0.9853138]
2401번째 cost:7.3365554271731526e-06, w:[1.0031383], b:[0.9928659]
2701번째 cost:1.7316773437414668e-06, w:[1.0015248], b:[0.99653405]
3001번째 cost:4.091789378435351e-07, w:[1.0007414], b:[0.9983153]
3301번째 cost:9.688102409199928e-08, w:[1.0003608], b:[0.99918026]
3601번째 cost:2.3044369967806233e-08, w:[1.000176], b:[0.99960005]
3901번째 cost:5.527212465494813e-09, w:[1.000086], b:[0.999804]
4201번째 cost:1.3694952949450112e-09, w:[1.0000424], b:[0.9999025]
4501번째 cost:3.1768840336177107e-10, w:[1.000020

In [40]:
w_, b_ = sess.run([w[0], b[0]])
w_, b_

(1.0000087, 0.9999811)

In [41]:
def predict(x):
    return x*w_ + b_

In [42]:
predict(5)

6.000024616718292

## 2.2 predict을 위한 placeholder이용
- placeholder : 외부에서 데이터를 입력받을 수 있는 노드

In [43]:
x = tf.placeholder(dtype=np.float32)
H = w_*x + b_
sess = tf.Session()
sess.run([H, x], {x:2.5})

[3.5000029, array(2.5, dtype=float32)]

In [44]:
sess.run(H, {x: np.array([2, 3, 3])})

array([3.5000029, 4.000007 , 4.5000114], dtype=float32)

In [47]:
# tensor 그래프 정의
# 데이터 셋 확보
x_data = np.array([1,2,3])
y_data = np.array([2,3,4])
# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)
# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습 목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
train = optimizer.minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 6001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b],
                                        feed_dict={x:x_data, y:y_data})
    if step%300==1:
        print(f'{step}번째 cost:{cost_val}, w:{w_val}, b:{b_val}')
print(f'{step}번째 cost:{cost_val}, w:{w_val}, b:{b_val}')

1번째 cost:34.611995697021484, w:[-0.9532895], b:[-0.07502083]
301번째 cost:0.0011116197565570474, w:[1.0386304], b:[0.9121838]
601번째 cost:0.000262293906416744, w:[1.0187649], b:[0.9573429]
901번째 cost:6.189199484651908e-05, w:[1.0091152], b:[0.979279]
1201번째 cost:1.4604219359171111e-05, w:[1.0044278], b:[0.98993456]
1501번째 cost:3.4464592317817733e-06, w:[1.0021511], b:[0.99511015]
1801번째 cost:8.139681995089632e-07, w:[1.0010455], b:[0.9976238]
2101번째 cost:1.9252827598847944e-07, w:[1.0005085], b:[0.99884444]
2401번째 cost:4.571311862378025e-08, w:[1.0002481], b:[0.999437]
2701번째 cost:1.090238299639168e-08, w:[1.0001209], b:[0.9997253]
3001번째 cost:2.6301070210621447e-09, w:[1.0000596], b:[0.9998652]
3301번째 cost:6.209243674781817e-10, w:[1.0000287], b:[0.9999345]
3601번째 cost:1.698860357945975e-10, w:[1.0000151], b:[0.999966]
3901번째 cost:5.195962063386794e-11, w:[1.0000086], b:[0.9999813]
4201번째 cost:5.195962063386794e-11, w:[1.0000086], b:[0.9999813]
4501번째 cost:5.195962063386794e-11, w:[1.000

In [48]:
# 예측하기
sess.run(H, feed_dict={x:2.5})

array([3.5000029], dtype=float32)

In [50]:
sess.run(H, feed_dict={x:np.array([2.5, 3.5])})

array([3.5000029, 4.5000114], dtype=float32)

## 2.3 scale이 다른 데이터들의 회귀분석 구현(scale조정X)

In [55]:
# tensor 그래프 정의
# 데이터 셋 확보
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])
# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)
# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습 목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
# optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
# train = optimizer.minimize(cost)
train = tf.train.GradientDescentOptimizer(learning_rate=0.1).minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 6001):
    _, cost_val, w_val, b_val = sess.run([train, cost, w, b],
                                        feed_dict={x:x_data, y:y_data})
    if step%300==1:
        print(f'{step}번째 cost:{cost_val}')
print(f'{step}번째 cost:{cost_val}')

1번째 cost:4504.3212890625
301번째 cost:79.16798400878906
601번째 cost:79.14022827148438
901번째 cost:79.13947296142578
1201번째 cost:79.13943481445312
1501번째 cost:79.13945770263672
1801번째 cost:79.13944244384766
2101번째 cost:79.13947296142578
2401번째 cost:79.13946533203125
2701번째 cost:79.13946533203125
3001번째 cost:79.13946533203125
3301번째 cost:79.13946533203125
3601번째 cost:79.13946533203125
3901번째 cost:79.13946533203125
4201번째 cost:79.13946533203125
4501번째 cost:79.13946533203125
4801번째 cost:79.13946533203125
5101번째 cost:79.13946533203125
5401번째 cost:79.13946533203125
5701번째 cost:79.13946533203125
6000번째 cost:79.13946533203125


## 2.4 scale이 다른 데이터의 회귀분석(scale조정 O)
### scale 조정방법 : 모든 데이터를 일정범위내로 조정
- normalization(정규화) : 모든 데이터를 0~1 사이로 조정
                       X - Xmin
    normalization = ───────────────
                      Xmax - Xmin
       
       * 위의 식보다 라이브러리 추천(sklearn.preprocessing.MinMaxScaler)
- standarization(표준화) : 데이터의 평균을 0, 표준편차를 1로 조정
                           X - Xmean
      standardization = ───────────────
                          Xstd(표준편차)
      * 위의 식보다 라이브러리 추천(sklearn.preprocessing.StandardScaler)

In [6]:
# 라이브러리를 쓰지 않고 정규화
x_data = np.array([1, 2, 5, 8, 10])
y_data = np.array([5, 15, 68, 80, 95])
norm_scaled_x_data = (x_data - x_data.min()) / (x_data.max() - x_data.min())
norm_scaled_y_data = (y_data - y_data.min()) / (y_data.max() - y_data.min())
print(norm_scaled_x_data)
print(norm_scaled_y_data)

[0.         0.11111111 0.44444444 0.77777778 1.        ]
[0.         0.11111111 0.7        0.83333333 1.        ]


In [7]:
# 라이브러리를 사용하여 정규화
from sklearn.preprocessing import MinMaxScaler, StandardScaler
x_data = np.array([1, 2, 5, 8, 10]).reshape(-1, 1)
y_data = np.array([5, 15, 68, 80, 95]).reshape(-1, 1)
scaler_x = MinMaxScaler() # x_data를 변환시킬 객체
scaler_x.fit(x_data)
norm_scaled_x_data = scaler_x.transform(x_data)
scaler_y = MinMaxScaler() # y_data를 변환시킬 객체
# scaler_y.fit(y_data)
# norm_scaled_y_data = scaler_y.transform(y_data)
norm_scaled_y_data = scaler_y.fit_transform(y_data)
np.column_stack([x_data, norm_scaled_x_data, y_data, norm_scaled_y_data])

array([[ 1.        ,  0.        ,  5.        ,  0.        ],
       [ 2.        ,  0.11111111, 15.        ,  0.11111111],
       [ 5.        ,  0.44444444, 68.        ,  0.7       ],
       [ 8.        ,  0.77777778, 80.        ,  0.83333333],
       [10.        ,  1.        , 95.        ,  1.        ]])

In [8]:
# 라이브러리를 쓰지 않고 표준화
x_data = np.array([1, 2, 5, 8, 10])
y_data = np.array([5, 15, 68, 80, 95])
stan_scaled_x_data = ( x_data - x_data.mean() ) / x_data.std()
stan_scaled_y_data = ( y_data - y_data.mean() ) / y_data.std()
print(np.column_stack([x_data, stan_scaled_x_data, norm_scaled_x_data]))
print()
print(np.column_stack([y_data, stan_scaled_y_data, norm_scaled_y_data]))

[[ 1.         -1.22474487  0.        ]
 [ 2.         -0.93313895  0.11111111]
 [ 5.         -0.05832118  0.44444444]
 [ 8.          0.81649658  0.77777778]
 [10.          1.39970842  1.        ]]

[[ 5.         -1.32373476  0.        ]
 [15.         -1.04563922  0.11111111]
 [68.          0.42826713  0.7       ]
 [80.          0.76198177  0.83333333]
 [95.          1.17912508  1.        ]]


In [9]:
# 라이브러리를 사용하여 표준화
x_data = np.array([1, 2, 5, 8, 10]).reshape(-1, 1)
y_data = np.array([5, 15, 68, 80, 95]).reshape(-1, 1)
scaler_x = StandardScaler()
stan_scaled_x_data = scaler_x.fit_transform(x_data)
scaler_y = StandardScaler()
stan_scaled_y_data = scaler_y.fit_transform(y_data)
np.column_stack([stan_scaled_x_data, stan_scaled_y_data])

array([[-1.22474487, -1.32373476],
       [-0.93313895, -1.04563922],
       [-0.05832118,  0.42826713],
       [ 0.81649658,  0.76198177],
       [ 1.39970842,  1.17912508]])

In [10]:
# 스케일 조정된 데이터를 다시 복구 : inverse_transform() 이용
scaler_x.inverse_transform(stan_scaled_x_data)

array([[ 1.],
       [ 2.],
       [ 5.],
       [ 8.],
       [10.]])

In [11]:
scaler_y.inverse_transform(stan_scaled_y_data)

array([[ 5.],
       [15.],
       [68.],
       [80.],
       [95.]])

In [12]:
# 데이터 셋 확보
x_data = np.array([1,2,5,8,10])
y_data = np.array([5,15,68,80,95])
# placeholder 노드 설정
x = tf.placeholder(dtype=tf.float32)
y = tf.placeholder(dtype=tf.float32)
# weight와 bias
w = tf.Variable( tf.random.normal([1]), name='weight' )
b = tf.Variable( tf.random.normal([1]), name='bias')
# hat, hypothesis : 결과는 numpy배열
H = w * x + b
# cost function (손실함수 : mse) : H-y의 제곱의 평균
cost = tf.reduce_mean(tf.square(H-y))
'''
학습 목적 : cost가 최소가 되는 w와 b를 찾는 것
cost함수가 2차함수이므로 곡선 그래프, 곡선 위 미분값이 0이 되는 방향 학습(경사하강법:GradientDescent)
'''
# optimizer = tf.train.GradientDescentOptimizer(learning_rate=0.01)
# train = optimizer.minimize(cost)
train = tf.train.GradientDescentOptimizer(learning_rate=0.1).minimize(cost)
# 세션 생성
sess = tf.Session()
# w와 b 초기화
sess.run(tf.global_variables_initializer())
# 학습(v2에서의 fit함수)
for step in range(1, 6001):
    _, cost_val = sess.run([train, cost],
                        feed_dict={x:stan_scaled_x_data,
                                   y:stan_scaled_y_data})
    if step%300==1:
        print(f'{step}번째 cost:{cost_val}')
print(f'{step}번째 cost:{cost_val}')

1번째 cost:4.784360408782959
301번째 cost:0.06120417267084122
601번째 cost:0.06120417267084122
901번째 cost:0.06120417267084122
1201번째 cost:0.06120417267084122
1501번째 cost:0.06120417267084122
1801번째 cost:0.06120417267084122
2101번째 cost:0.06120417267084122
2401번째 cost:0.06120417267084122
2701번째 cost:0.06120417267084122
3001번째 cost:0.06120417267084122
3301번째 cost:0.06120417267084122
3601번째 cost:0.06120417267084122
3901번째 cost:0.06120417267084122
4201번째 cost:0.06120417267084122
4501번째 cost:0.06120417267084122
4801번째 cost:0.06120417267084122
5101번째 cost:0.06120417267084122
5401번째 cost:0.06120417267084122
5701번째 cost:0.06120417267084122
6000번째 cost:0.061204176396131516


# 2.5 독립변수 x가 3개, 타겟변수 y가 1개인 회귀분석